In [51]:
import requests
import json
import os
import time
import logging
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv

Obtenir le token qui se rafraichit toutes les 15 minutes

In [52]:
load_dotenv()
username = os.getenv("ENTSOE_USER")
password = os.getenv("ENTSOE_PWD")

In [53]:
keycloak_url = "https://keycloak.tp.entsoe.eu/realms/tp/protocol/openid-connect/token"

payload = {
    "client_id": "tp-fms-public",
    "grant_type": "password",
    "username": username,
    "password": password
}

headers = {
    'Content-Type': 'application/x-www-form-urlencoded'
}

response = requests.post(keycloak_url, headers=headers, data=payload)
access_token = response.json().get("access_token")
print(access_token)

eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJHdzVDNVJDRm9pTVVlenY4UF9ScGxzRk83Nm1hTk05bjNKck9CcWV1OVV3In0.eyJleHAiOjE3OTExODMwMTksImlhdCI6MTc5MTE4MjExOSwianRpIjoiYmJjOGYzMzktOTUyZC00MzA2LWFkOTktZTA2NjIxZjc2ZjhlIiwiaXNzIjoiaHR0cHM6Ly9rZXljbG9hay50cC5lbnRzb2UuZXUvcmVhbG1zL3RwIiwiYXVkIjpbInVzeS10cC11c2VyZzAxIiwidXN5LXRwLWFyY2hpdmluZ2cwMSIsInRwLXdlYiIsImFjY291bnQiXSwic3ViIjoiYmNlMzkzYmUtYjVkOS00ZWM1LWEzZDctNWUwOTNhYzEwZWExIiwidHlwIjoiQmVhcmVyIiwiYXpwIjoidHAtZm1zLXB1YmxpYyIsInNpZCI6IjFkNjE1ZDNhLTFhNzQtNDVhYS05N2ZhLTE1NzMxMDUyM2ExZSIsImFjciI6IjEiLCJhbGxvd2VkLW9yaWdpbnMiOlsiLyoiXSwicmVhbG1fYWNjZXNzIjp7InJvbGVzIjpbIm9mZmxpbmVfYWNjZXNzIiwidW1hX2F1dGhvcml6YXRpb24iLCJkZWZhdWx0LXJvbGVzLXRwIl19LCJyZXNvdXJjZV9hY2Nlc3MiOnsidHAtd2ViIjp7InJvbGVzIjpbIlJlZ2lzdGVyZWQgdXNlciJdfSwiYWNjb3VudCI6eyJyb2xlcyI6WyJtYW5hZ2UtYWNjb3VudCIsIm1hbmFnZS1hY2NvdW50LWxpbmtzIiwidmlldy1wcm9maWxlIl19fSwic2NvcGUiOiJ1c3ktdHAtdXNlcmcwMSBwcm9maWxlIHVzeS10cC1hcmNoaXZpbmdnMDEgZW1haWwiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwicHJlZmVycmVk

In [ ]:
folders  = [
    "ActualTotalLoad_6.1.A_r3",
    "AggregatedGenerationPerType_16.1.B_C_r3",
    "InstalledGenerationCapacityAggregated_14.1.A_r3",
    "PhysicalFlows_12.1.G_r3",
    "ProductionAndGenerationUnits_r3",
    "ProductionAndGenerationUnits_r3.1",
    "InstalledGenerationCapacityPerProductionUnit_14.1.B_r3",
    "InstalledGenerationCapacityPerProductionUnit_14.1.B_r3.1",
]

In [56]:

folder_url = "https://fms.tp.entsoe.eu/listFolder"
dl_url = "https://fms.tp.entsoe.eu/downloadFileContent"

# ============================================================
# Gestion du token : rafraîchi automatiquement avant expiration   <--- créer par IA
# ============================================================
TOKEN_DUREE_MAX = 8 * 60  # rafraîchit après 8 min, avant l'expiration à 10 min

_token_cache = {"access_token": None, "obtenu_a": 0}


def obtenir_token():
    payload_auth = {
        "client_id": "tp-fms-public",
        "grant_type": "password",
        "username": username,
        "password": password,
    }
    headers_auth = {'Content-Type': 'application/x-www-form-urlencoded'}
    resp_auth = requests.post(keycloak_url, headers=headers_auth, data=payload_auth)
    resp_auth.raise_for_status()
    return resp_auth.json()["access_token"]


def get_headers():
    maintenant = time.time()
    if _token_cache["access_token"] is None or (maintenant - _token_cache["obtenu_a"]) > TOKEN_DUREE_MAX:
        _token_cache["access_token"] = obtenir_token()
        _token_cache["obtenu_a"] = maintenant
    return {
        'Authorization': f'Bearer {_token_cache["access_token"]}',
        'Content-Type': 'application/json',
        'Accept': '*/*',
    }

# ============================================================
# jusqu'ici   <--- créer par IA
# ============================================================

metriques = []

# ============================================================
# Cette boucle passe dans chaque dossiers pour télécharger chaque fichiers 
# ============================================================
path_datas = Path(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Electricite")
for folder in folders:
    folder_name = folder
    path_folder = Path(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Electricite\{folder_name}")
    if not os.path.exists(path_folder): os.makedirs(path_folder)

    payload = json.dumps({
        "path": f"/TP_export/{folder_name}/",  
        "pageInfo": {
            "pageIndex": 0, "pageSize": 5000
            },
    })
    resp = requests.post(folder_url, headers=get_headers(), data=payload)
    files = resp.json()["contentItemList"]

    for file in files:
        file_name = file["name"]
        path_file = Path(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Electricite\{folder_name}\{file_name}")
        
        debut = time.perf_counter()

        payload_dl = json.dumps({
            "folder": f"/TP_export/{folder_name}", 
            "filename": file_name,
            "topLevelFolder": "TP_export",
            "downloadAsZip": False,
        })
        resp_dl = requests.post(dl_url, headers=get_headers(), data=payload_dl)

        with open(path_file, "wb") as f:
            f.write(resp_dl.content)

        duree = time.perf_counter() - debut
        taille_ko = os.path.getsize(path_file) / 1024
        lines = resp_dl.content.count(b"\n")
        
        metriques.append({"dossier":folder_name, "fichier": file_name,
                           "taille_ko": taille_ko, "duree_s": duree, "lignes":lines})

pd.DataFrame(metriques).to_csv(rf"C:\Users\PC\Documents\Projets\TFE\Datas\Electricite\metriques_electricite_ingestion.csv", index=False)

KeyboardInterrupt: 